# Topic 2: Spark SQL & Views

> **Phase 3 → Week 5 → Topic 2**

---

## SQL and DataFrame API — Same Engine, Same Plan

```
DataFrame API                     Spark SQL
─────────────────────────────     ─────────────────────────────
df.groupBy("dept")                SELECT dept, AVG(salary)
  .agg(F.avg("salary"))      ≡   FROM employees
  .orderBy("dept")               GROUP BY dept
                                  ORDER BY dept

Both produce the SAME logical plan → same physical plan → same execution
```

The DataFrame API and Spark SQL are **two syntaxes for the same thing**. Catalyst compiles both to an identical logical plan. You choose based on preference — SQL is great for ad-hoc analysis, DataFrame API for programmatic pipelines.

---

## View Types

| View Type | Scope | Lifetime | Method |
|-----------|-------|---------|--------|
| Temp View | Current SparkSession | Until session ends | `createOrReplaceTempView("name")` |
| Global Temp View | ALL SparkSessions in same app | Until app ends | `createOrReplaceGlobalTempView("name")` |
| Hive Table | Permanent | Persists across sessions | `df.write.saveAsTable("name")` (requires Hive metastore) |

**Global views** are accessed with the `global_temp` prefix: `SELECT * FROM global_temp.my_view`.

---

## Interview Cheat Sheet

**Q: What's the difference between createTempView and createOrReplaceTempView?**
> `createTempView()` throws an `AnalysisException` if a view with that name already exists. `createOrReplaceTempView()` silently overwrites the existing view. In production code, always use `createOrReplaceTempView` to avoid brittle code that breaks on second run.

**Q: Does Spark SQL perform the same as the DataFrame API?**
> Yes — exactly the same. Both are compiled by Catalyst into a logical plan, which is optimized identically. The only difference is syntax. Choose SQL for readability when the query is complex; choose DataFrame API when you need to build queries programmatically.

**Q: What is a global temporary view?**
> A global temporary view is visible across all SparkSessions within the same Spark application (same JVM process). It's stored in the `global_temp` database. Access it with `SELECT * FROM global_temp.view_name`. Useful for sharing intermediate results between different sessions in the same application.

**Q: What advanced SQL features does Spark SQL support?**
> Spark SQL supports window functions (`OVER`/`PARTITION BY`), CTEs (`WITH`), subqueries, lateral views, PIVOT, UNPIVOT, ROLLUP, CUBE, GROUPING SETS, and all standard JOIN types. It also supports ANSI SQL compliance mode.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week5 - Spark SQL") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
products  = spark.read.csv("/workspace/week4/data/products.csv",  header=True, inferSchema=True)
sales     = spark.read.csv("/workspace/week4/data/sales.csv",     header=True, inferSchema=True)

print("Data loaded")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/14 03:45:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data loaded


## Part 1: Registering Views and Running SQL

In [2]:
# Register all tables as temp views
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
sales.createOrReplaceTempView("sales")

# List all registered views
print("Registered temp views:")
spark.catalog.listTables()
for t in spark.catalog.listTables():
    print(f"  {t.name} ({t.tableType})")

Registered temp views:


  customers (TEMPORARY)
  orders (TEMPORARY)
  products (TEMPORARY)
  sales (TEMPORARY)


In [3]:
# Basic Spark SQL
result = spark.sql("""
    SELECT 
        c.name         AS customer_name,
        c.country,
        COUNT(o.order_id)   AS total_orders,
        SUM(o.amount)       AS total_spent,
        ROUND(AVG(o.amount), 2) AS avg_order
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name, c.country
    ORDER BY total_spent DESC NULLS LAST
""")

print("Customer spend summary (SQL):")
result.show()

Customer spend summary (SQL):


+--------------+-----------+------------+------------------+---------+
| customer_name|    country|total_orders|       total_spent|avg_order|
+--------------+-----------+------------+------------------+---------+
|   James Brown|         UK|           2|2899.9700000000003|  1449.99|
|Carol Williams|        USA|           3|           1999.95|   666.65|
|  Alice Sharma|      India|           3|           1579.96|   526.65|
|  Frank Tanaka|      Japan|           2|            569.96|   284.98|
|    Eve Müller|    Germany|           2|244.98000000000002|   122.49|
|      Bob Chen|      China|           2|            209.95|   104.98|
|  Henry Okafor|    Nigeria|           1|            149.95|   149.95|
|    Dave Kumar|      India|           2|            109.97|    54.99|
|   Irene Rossi|      Italy|           1|             69.98|    69.98|
|    Grace Park|South Korea|           1|             44.99|    44.99|
+--------------+-----------+------------+------------------+---------+



In [4]:
# Identical query using DataFrame API — produces the SAME plan
result_df = customers.alias("c") \
    .join(orders.alias("o"), on="customer_id", how="left") \
    .groupBy("c.name", "c.country") \
    .agg(
        F.count("o.order_id").alias("total_orders"),
        F.sum("o.amount").alias("total_spent"),
        F.round(F.avg("o.amount"), 2).alias("avg_order")
    ) \
    .orderBy(F.col("total_spent").desc_nulls_last())

print("Identical result from DataFrame API:")
result_df.show()

# Verify same plan
print("\nSQL plan:")
result.explain()
print("\nDataFrame API plan:")
result_df.explain()

Identical result from DataFrame API:


+--------------+-----------+------------+------------------+---------+
|          name|    country|total_orders|       total_spent|avg_order|
+--------------+-----------+------------+------------------+---------+
|   James Brown|         UK|           2|2899.9700000000003|  1449.99|
|Carol Williams|        USA|           3|           1999.95|   666.65|
|  Alice Sharma|      India|           3|           1579.96|   526.65|
|  Frank Tanaka|      Japan|           2|            569.96|   284.98|
|    Eve Müller|    Germany|           2|244.98000000000002|   122.49|
|      Bob Chen|      China|           2|            209.95|   104.98|
|  Henry Okafor|    Nigeria|           1|            149.95|   149.95|
|    Dave Kumar|      India|           2|            109.97|    54.99|
|   Irene Rossi|      Italy|           1|             69.98|    69.98|
|    Grace Park|South Korea|           1|             44.99|    44.99|
+--------------+-----------+------------+------------------+---------+


SQL 

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_spent#272 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_spent#272 DESC NULLS LAST, 4), ENSURE_REQUIREMENTS, [plan_id=472]
      +- HashAggregate(keys=[name#18, country#20], functions=[count(order_id#46), sum(amount#52), avg(amount#52)])
         +- Exchange hashpartitioning(name#18, country#20, 4), ENSURE_REQUIREMENTS, [plan_id=469]
            +- HashAggregate(keys=[name#18, country#20], functions=[partial_count(order_id#46), partial_sum(amount#52), partial_avg(amount#52)])
               +- Project [name#18, country#20, order_id#46, amount#52]
                  +- BroadcastHashJoin [customer_id#17], [customer_id#47], LeftOuter, BuildRight, false
                     :- FileScan csv [customer_id#17,name#18,country#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/week4/data/customers.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: st

## Part 2: Advanced SQL Features

In [5]:
# CTEs (Common Table Expressions) — WITH clause
cte_result = spark.sql("""
    WITH
    customer_spend AS (
        SELECT customer_id, SUM(amount) AS total_spent, COUNT(*) AS order_count
        FROM orders
        WHERE status = 'delivered'
        GROUP BY customer_id
    ),
    high_value AS (
        SELECT customer_id, total_spent, order_count
        FROM customer_spend
        WHERE total_spent > 500
    )
    SELECT c.name, c.country, c.tier, h.total_spent, h.order_count
    FROM customers c
    INNER JOIN high_value h ON c.customer_id = h.customer_id
    ORDER BY h.total_spent DESC
""")

print("CTEs — high value customers:")
cte_result.show()

CTEs — high value customers:


+--------------+-------+----+-----------+-----------+
|          name|country|tier|total_spent|order_count|
+--------------+-------+----+-----------+-----------+
|   James Brown|     UK|Gold|    2599.98|          1|
|Carol Williams|    USA|Gold|    1899.97|          2|
|  Alice Sharma|  India|Gold|    1579.96|          3|
|  Frank Tanaka|  Japan|Gold|     569.96|          2|
+--------------+-------+----+-----------+-----------+



In [6]:
# Window functions in SQL
window_sql = spark.sql("""
    SELECT
        salesperson,
        region,
        month,
        revenue,
        RANK() OVER (PARTITION BY salesperson ORDER BY revenue DESC) AS revenue_rank,
        SUM(revenue) OVER (PARTITION BY salesperson ORDER BY 
            CASE month
                WHEN 'January' THEN 1 WHEN 'February' THEN 2
                WHEN 'March'   THEN 3 WHEN 'April'    THEN 4
                WHEN 'May'     THEN 5 ELSE 6
            END
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
        LAG(revenue, 1) OVER (PARTITION BY salesperson ORDER BY 
            CASE month
                WHEN 'January' THEN 1 WHEN 'February' THEN 2
                WHEN 'March'   THEN 3 WHEN 'April'    THEN 4
                WHEN 'May'     THEN 5 ELSE 6
            END
        ) AS prev_revenue
    FROM sales
    WHERE salesperson = 'Alice'
    ORDER BY revenue_rank
""")

print("Window functions in SQL (Alice's data):")
window_sql.show()

Window functions in SQL (Alice's data):


+-----------+------+--------+-------+------------+-------------+------------+
|salesperson|region|   month|revenue|revenue_rank|running_total|prev_revenue|
+-----------+------+--------+-------+------------+-------------+------------+
|      Alice| North|     May|  25000|           1|        92000|       13000|
|      Alice| North|   March|  22000|           2|        54000|       17000|
|      Alice| North|    June|  19000|           3|       111000|       25000|
|      Alice| North|February|  17000|           4|        32000|       15000|
|      Alice| North| January|  15000|           5|        15000|        NULL|
|      Alice| North|   April|  13000|           6|        67000|       22000|
+-----------+------+--------+-------+------------+-------------+------------+



In [7]:
# Subqueries — correlated and non-correlated
subquery_result = spark.sql("""
    SELECT name, country, tier
    FROM customers
    WHERE customer_id IN (
        SELECT DISTINCT customer_id
        FROM orders
        WHERE amount > 500
    )
    ORDER BY name
""")
print("Customers with at least one order > 500 (subquery):")
subquery_result.show()

# Scalar subquery
scalar_result = spark.sql("""
    SELECT
        order_id,
        amount,
        ROUND(amount / (SELECT AVG(amount) FROM orders), 2) AS pct_of_avg
    FROM orders
    ORDER BY pct_of_avg DESC
    LIMIT 5
""")
print("Top orders as % of average (scalar subquery):")
scalar_result.show()

Customers with at least one order > 500 (subquery):


+--------------+-------+----+
|          name|country|tier|
+--------------+-------+----+
|  Alice Sharma|  India|Gold|
|Carol Williams|    USA|Gold|
|   James Brown|     UK|Gold|
+--------------+-------+----+

Top orders as % of average (scalar subquery):


+--------+-------+----------+
|order_id| amount|pct_of_avg|
+--------+-------+----------+
|    O012|2599.98|      5.66|
|    O001|1299.99|      2.83|
|    O004|1299.99|      2.83|
|    O016|1299.99|      2.83|
|    O005| 599.98|      1.31|
+--------+-------+----------+



In [8]:
# PIVOT in Spark SQL
# Rotate rows into columns — classic reporting pattern

pivot_result = spark.sql("""
    SELECT *
    FROM (
        SELECT salesperson, month, revenue
        FROM sales
        WHERE month IN ('January', 'February', 'March')
    )
    PIVOT (
        SUM(revenue)
        FOR month IN ('January' AS jan, 'February' AS feb, 'March' AS mar)
    )
    ORDER BY salesperson
""")

print("PIVOT — months as columns:")
pivot_result.show()

# Same thing in DataFrame API
print("Same using DataFrame .pivot():")
sales.filter(F.col("month").isin("January", "February", "March")) \
     .groupBy("salesperson") \
     .pivot("month", ["January", "February", "March"]) \
     .agg(F.sum("revenue")) \
     .orderBy("salesperson") \
     .show()

PIVOT — months as columns:


+-----------+-----+-----+-----+
|salesperson|  jan|  feb|  mar|
+-----------+-----+-----+-----+
|      Alice|15000|17000|22000|
|        Bob|12000|11000|19000|
|      Carol|18000|20000|16000|
|       Dave| 9000|14000|21000|
+-----------+-----+-----+-----+

Same using DataFrame .pivot():


+-----------+-------+--------+-----+
|salesperson|January|February|March|
+-----------+-------+--------+-----+
|      Alice|  15000|   17000|22000|
|        Bob|  12000|   11000|19000|
|      Carol|  18000|   20000|16000|
|       Dave|   9000|   14000|21000|
+-----------+-------+--------+-----+



In [9]:
# ROLLUP and CUBE — hierarchical aggregations
rollup_result = spark.sql("""
    SELECT
        COALESCE(region, 'ALL REGIONS') AS region,
        COALESCE(salesperson, 'ALL SALESPEOPLE') AS salesperson,
        SUM(revenue) AS total_revenue
    FROM sales
    GROUP BY ROLLUP(region, salesperson)
    ORDER BY region, salesperson
""")

print("ROLLUP — subtotals per region and grand total:")
rollup_result.show()

ROLLUP — subtotals per region and grand total:


+-----------+---------------+-------------+
|     region|    salesperson|total_revenue|
+-----------+---------------+-------------+
|ALL REGIONS|ALL SALESPEOPLE|       422000|
|       East|ALL SALESPEOPLE|       105000|
|       East|          Carol|       105000|
|      North|ALL SALESPEOPLE|       111000|
|      North|          Alice|       111000|
|      South|ALL SALESPEOPLE|       103000|
|      South|            Bob|       103000|
|       West|ALL SALESPEOPLE|       103000|
|       West|           Dave|       103000|
+-----------+---------------+-------------+



## Part 3: Global Temp Views

In [10]:
# Global temp view — accessible from any SparkSession
orders.createOrReplaceGlobalTempView("global_orders")

# Access with global_temp prefix
spark.sql("SELECT COUNT(*) FROM global_temp.global_orders").show()

# Create a second SparkSession and access the global view
spark2 = SparkSession.builder \
    .appName("Session2") \
    .master("local[2]") \
    .getOrCreate()

print("Accessing global view from a different SparkSession:")
spark2.sql("SELECT COUNT(*) as count FROM global_temp.global_orders").show()

# Temp view is NOT visible from spark2
try:
    spark2.sql("SELECT COUNT(*) FROM orders")
except Exception as e:
    print(f"Temp view 'orders' NOT visible from spark2: {type(e).__name__}")

+--------+
|count(1)|
+--------+
|      20|
+--------+

Accessing global view from a different SparkSession:


26/05/14 03:46:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-----+
|count|
+-----+
|   20|
+-----+



## Part 4: Catalog API — Inspecting the Spark Metastore

In [11]:
# spark.catalog — inspect registered tables, databases, columns

print("Current database:", spark.catalog.currentDatabase())
print()

print("All registered tables:")
for t in spark.catalog.listTables():
    print(f"  {t.name:20s} | type={t.tableType:20s} | isTemp={t.isTemporary}")
print()

print("Columns of 'orders' view:")
for col in spark.catalog.listColumns("orders"):
    print(f"  {col.name:15s} | type={col.dataType}")

Current database: default

All registered tables:


  customers            | type=TEMPORARY            | isTemp=True
  orders               | type=TEMPORARY            | isTemp=True
  products             | type=TEMPORARY            | isTemp=True
  sales                | type=TEMPORARY            | isTemp=True

Columns of 'orders' view:


  order_id        | type=string
  customer_id     | type=int
  product_id      | type=string
  quantity        | type=int
  order_date      | type=date
  status          | type=string
  amount          | type=double


In [12]:
# Manage views

# Check if a view exists before dropping
print(f"'orders' view exists: {spark.catalog.tableExists('orders')}")
print(f"'nonexistent' exists: {spark.catalog.tableExists('nonexistent')}")

# Drop a temp view
spark.catalog.dropTempView("customers")
print(f"'customers' after drop: {spark.catalog.tableExists('customers')}")

# Re-register it
customers.createOrReplaceTempView("customers")
print(f"'customers' after re-register: {spark.catalog.tableExists('customers')}")

'orders' view exists: True


'nonexistent' exists: False
'customers' after drop: False
'customers' after re-register: True


## Exercises

1. Write a SQL CTE that:
   - First computes total revenue per product category
   - Then ranks categories by revenue
   - Shows only top 2 categories

2. Write the same query as Exercise 1 using only the DataFrame API.

3. Use PIVOT in SQL to show total revenue per salesperson per month (columns = months Jan-Jun).

4. Register a global temp view for the `products` table. Create a new SparkSession and query it.

5. Using `spark.catalog`, list all tables and their types. Then drop and re-register the `sales` view.

In [13]:
# Exercise 1: CTE for top 2 categories
spark.sql("""
    WITH cat_revenue AS (
        SELECT p.category, SUM(o.amount) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.category
    ),
    ranked AS (
        SELECT *, RANK() OVER (ORDER BY revenue DESC) AS rnk
        FROM cat_revenue
    )
    SELECT category, revenue, rnk
    FROM ranked
    WHERE rnk <= 2
""").show()

26/05/14 03:46:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/14 03:46:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/14 03:46:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/14 03:46:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+-----------------+---+
|   category|          revenue|rnk|
+-----------+-----------------+---+
|Electronics|7349.799999999997|  1|
|  Furniture|          1469.94|  2|
+-----------+-----------------+---+



In [14]:
# Exercise 3: PIVOT — revenue per salesperson per month
spark.sql("""
    SELECT *
    FROM (
        SELECT salesperson, month, revenue FROM sales
    )
    PIVOT (
        SUM(revenue)
        FOR month IN (
            'January' AS jan, 'February' AS feb, 'March' AS mar,
            'April' AS apr, 'May' AS may, 'June' AS jun
        )
    )
    ORDER BY salesperson
""").show()

+-----------+-----+-----+-----+-----+-----+-----+
|salesperson|  jan|  feb|  mar|  apr|  may|  jun|
+-----------+-----+-----+-----+-----+-----+-----+
|      Alice|15000|17000|22000|13000|25000|19000|
|        Bob|12000|11000|19000|24000|16000|21000|
|      Carol|18000|20000|16000|11000|23000|17000|
|       Dave| 9000|14000|21000|18000|15000|26000|
+-----------+-----+-----+-----+-----+-----+-----+

